In [1]:
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("CUDA:", torch.cuda.is_available())
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

GPU: NVIDIA A40
BF16 supported: True
CUDA: True
VRAM: 47.7 GB


In [2]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets huggingface_hub
print("Installed.")

Installed.


In [3]:
import os
print([f for f in os.listdir('.') if f.endswith('.jsonl')])

['train.jsonl', 'val.jsonl', 'test.jsonl']


In [4]:
import os
print("JSONL files here:", [f for f in os.listdir('.') if f.endswith('.jsonl')])

# quick check they're readable
import json
train = [json.loads(l) for l in open('train.jsonl')]
print(f"Train examples: {len(train)}")
print("First example keys:", list(train[0].keys()))

JSONL files here: ['train.jsonl', 'val.jsonl', 'test.jsonl']
Train examples: 10509
First example keys: ['instruction', 'input', 'output', 'source']


In [ ]:
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from huggingface_hub import login

login(token="")  # your HF write token

MODEL_NAME = "google/gemma-2-2b-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # bf16 — the A40 supports it natively
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,                     # bf16 throughout
    attn_implementation="eager",              # recommended for gemma2
)
print("Loaded. Memory:", f"{torch.cuda.memory_allocated()/1e9:.2f} GB")
print("Dtype:", next(model.parameters()).dtype)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded. Memory: 2.23 GB
Dtype: torch.bfloat16


In [6]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Load + format data
def load_jsonl(p): return [json.loads(l) for l in open(p)]
train_data = load_jsonl('train.jsonl')
val_data = load_jsonl('val.jsonl')

def format_example(ex):
    user = ex['instruction']
    if ex.get('input','').strip(): user = f"{ex['instruction']}\n\n{ex['input']}"
    msgs = [{"role":"user","content":user},{"role":"assistant","content":ex['output']}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

train_formatted = Dataset.from_list([format_example(e) for e in train_data])
val_formatted = Dataset.from_list([format_example(e) for e in val_data])
print(f"Data ready: {len(train_formatted)} train, {len(val_formatted)} val")

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881
Data ready: 10509 train, 583 val


In [ ]:
from trl import SFTTrainer, SFTConfig

tiny = train_formatted.select(range(100))
tiny_trainer = SFTTrainer(
    model=model, train_dataset=tiny,
    args=SFTConfig(
        output_dir="./tiny", num_train_epochs=1,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        learning_rate=2e-4, max_length=768,
        bf16=True,                      # bf16 — A40 supports it
        logging_steps=2, save_strategy="no", report_to="none",
        optim="paged_adamw_8bit", dataset_text_field="text",
    ),
)
tiny_trainer.train()

# Immediately test generation
import torch
msgs = [{"role":"user","content":"मुझे चाय बनाने की विधि बताओ"}]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
with torch.no_grad():
    out = model.generate(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
        max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print("\n--- TINY OUTPUT ---")
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip())

In [ ]:
import torch

# Switch model to proper inference mode
model.config.use_cache = True
model.gradient_checkpointing_disable()
model.eval()

msgs = [{"role":"user","content":"मुझे चाय बनाने की विधि बताओ"}]
inputs = tokenizer.apply_chat_template(
    msgs, tokenize=True, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to("cuda")

with torch.no_grad():
    out = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=150,
        do_sample=False,
        use_cache=True,                      # explicitly enable cache
        pad_token_id=tokenizer.eos_token_id,
    )
print("--- OUTPUT (inference mode) ---")
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip())

In [10]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

# Custom callback that prints loss to stdout (streams reliably)
class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            print(f"Step {state.global_step}/{state.max_steps} | "
                  f"loss={logs['loss']:.4f} | "
                  f"epoch={logs.get('epoch', 0):.2f}", flush=True)
        if logs and 'eval_loss' in logs:
            print(f"  >>> VALIDATION at step {state.global_step}: "
                  f"eval_loss={logs['eval_loss']:.4f}", flush=True)

# Ensure proper training mode
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.train()

full_trainer = SFTTrainer(
    model=model,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    args=SFTConfig(
        output_dir="./gemma-hindi-lora",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.01,
        max_length=768,
        bf16=True,
        logging_steps=10,
        eval_strategy="steps", eval_steps=200,
        save_strategy="steps", save_steps=200, save_total_limit=2,
        optim="paged_adamw_8bit",
        report_to="none",
        dataset_text_field="text",
        seed=42,
    ),
    callbacks=[PrintLossCallback()],   # <-- the live logger
)
print("Launching full training with live logging...")
full_trainer.train()
print("\n✅ Done!")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/10509 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10509 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10509 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/583 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/583 [00:00<?, ? examples/s]

Launching full training with live logging...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.673211,1.589816,1.609749,1050016.000000,0.634139
400,1.512555,1.551554,1.557140,2109083.000000,0.641535
600,1.560843,1.530313,1.552091,3153690.000000,0.645373
800,1.341985,1.538743,1.379346,4203661.000000,0.645224
1000,1.302730,1.530033,1.354411,5265866.000000,0.647710
1200,1.336134,1.515979,1.390806,6316465.000000,0.650014
1400,1.135587,1.566676,1.223225,7350045.000000,0.647009


Step 10/1971 | loss=1.6275 | epoch=0.02
Step 20/1971 | loss=1.5920 | epoch=0.03
Step 30/1971 | loss=1.5493 | epoch=0.05
Step 40/1971 | loss=1.4521 | epoch=0.06
Step 50/1971 | loss=1.4533 | epoch=0.08
Step 60/1971 | loss=1.4024 | epoch=0.09
Step 70/1971 | loss=1.3978 | epoch=0.11
Step 80/1971 | loss=1.4452 | epoch=0.12
Step 90/1971 | loss=1.5985 | epoch=0.14
Step 100/1971 | loss=1.6304 | epoch=0.15
Step 110/1971 | loss=1.6036 | epoch=0.17
Step 120/1971 | loss=1.6097 | epoch=0.18
Step 130/1971 | loss=1.6139 | epoch=0.20
Step 140/1971 | loss=1.5790 | epoch=0.21
Step 150/1971 | loss=1.5371 | epoch=0.23
Step 160/1971 | loss=1.6083 | epoch=0.24
Step 170/1971 | loss=1.6149 | epoch=0.26
Step 180/1971 | loss=1.6222 | epoch=0.27
Step 190/1971 | loss=1.5733 | epoch=0.29
Step 200/1971 | loss=1.6732 | epoch=0.30
  >>> VALIDATION at step 200: eval_loss=1.5898
Step 210/1971 | loss=1.6181 | epoch=0.32
Step 220/1971 | loss=1.5875 | epoch=0.33
Step 230/1971 | loss=1.5880 | epoch=0.35
Step 240/1971 | los

KeyboardInterrupt: 

In [11]:
import torch
from peft import PeftModel

# Load the base model fresh, then attach your best adapter
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
base = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b-it", quantization_config=bnb_config,
    device_map="auto", dtype=torch.bfloat16, attn_implementation="eager",
)
ft_model = PeftModel.from_pretrained(base, "/workspace/best-checkpoint-1200")

# CRITICAL: inference mode (the use_cache fix)
ft_model.config.use_cache = True
ft_model.eval()

tok = AutoTokenizer.from_pretrained("/workspace/best-checkpoint-1200")

def gen(instruction, max_new_tokens=256):
    msgs = [{"role":"user","content":instruction}]
    inputs = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():
        out = ft_model.generate(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9,
            use_cache=True, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Test on a few prompts
for p in ["मुझे चाय बनाने की विधि बताओ",
          "yaar mujhe weekend ke liye ek movie suggest karo",
          "भारत की राजधानी क्या है और वहाँ क्या देखना चाहिए?"]:
    print("PROMPT:", p)
    print("RESPONSE:", gen(p))
    print("-"*50)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

PROMPT: मुझे चाय बनाने की विधि बताओ
RESPONSE: चाय को एक विशेष मर्तबान या चाय के बर्तन में डालकर और गर्म पानी डालकर बनाया जाता है। पानी को मर्तबान में डालने से पहले, चाय को मर्तबान में डालकर घुमाया जाता है, ताकि उसमें हवा की हवा हो। फिर, गर्म पानी मर्तबान में डालने के बाद, चाय को 3 से 5 मिनट के लिए गर्म होने के लिए छोड़ दिया जाता है। अंत में, चाय को चाय के बर्तन में डालकर, चाय के बर्तन को ढककर, और चाय को 5 से 10 मिनट के लिए गर्म होने के लिए छोड़ दिया जाता है। चाय को तब तक गरम रखा जाना चाहिए जब तक कि यह ठंडा न हो जाए।
--------------------------------------------------
PROMPT: yaar mujhe weekend ke liye ek movie suggest karo
RESPONSE: बिलकुल! आप इस वेबकैम पर 'द हार्ड रॉक' फिल्म को देख सकते हैं, जो एक 2019 की हॉकी फिल्म है। यह 2018 में विश्व हॉकी चैम्पियनशिप के दौरान माइकल 'मैट' ओ'सवाल के जीवन पर आधारित है, एक क्रिकेटर को हॉकी में परिवर्तित करने वाले एक आदमी की कहानी। इस फिल्म में प्रसिद्ध अभिनेता जैसे कि विलियम हॉग, जोसेफ ब्रूक्स, और जॉन थॉमस शामिल हैं। फिल्म में हॉकी के बारे में बहुत सार

In [12]:
ft_model.push_to_hub("naveenk879/gemma-2-2b-hindi-lora")
tok.push_to_hub("naveenk879/gemma-2-2b-hindi-lora")
print("Working model pushed to HF!")

README.md:   0%|          | 0.00/554 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Working model pushed to HF!


In [13]:
import json, random

def load_jsonl(p): return [json.loads(l) for l in open(p)]
test = load_jsonl('test.jsonl')

# Rebuild the same 300 by-source eval set (seed=42, 100 per source)
random.seed(42)
from collections import defaultdict
by_source = defaultdict(list)
for ex in test:
    by_source[ex['source']].append(ex)

eval_set = []
for source, examples in by_source.items():
    n = min(100, len(examples))
    for ex in random.sample(examples, n):
        e = dict(ex); e['eval_category'] = source
        eval_set.append(e)
random.shuffle(eval_set)

# The 20 hinglish prompts (same as before)
hinglish = [
    "yaar mujhe weekend ke liye ek acchi movie suggest karo",
    "bhai ek easy recipe batao dinner ke liye",
    "mujhe motivation chahiye, kuch accha bolo",
    "exam ke liye kaise padhu? tips do",
    "subah jaldi uthne ke liye kya karu?",
    "AI kya hota hai? simple language mein samjhao",
    "paise kaise bachaye? kuch practical tips do",
    "stress kam karne ke liye kya karu?",
    "achhi neend ke liye kya tips hai?",
    "interview ke liye kaise prepare karu?",
    "chai kaise banaye? step by step batao",
    "resume kaise banaye? guide karo",
    "ghar pe workout kaise kare bina equipment ke?",
    "dosti ke baare mein ek choti poem likho",
    "ek motivational quote banao jo students ke liye ho",
    "kya online padhai offline se behtar hai? apna view do",
    "social media acha hai ya bura? dono side batao",
    "mujhe Python seekhna hai, kahan se start karu?",
    "best budget phone kaunsa hai 2024 mein?",
    "weekend pe Jaipur mein ghumne ki jagah batao",
]
for h in hinglish:
    eval_set.append({"instruction": h, "input": "", "output": "", "eval_category": "hinglish"})

print(f"Eval set: {len(eval_set)} ({len([e for e in eval_set if e['eval_category']=='hinglish'])} hinglish)")

Eval set: 320 (20 hinglish)


In [14]:
import json, re, time
from collections import Counter

# Metric functions (same as baseline)
def hindi_ratio(text):
    if not text: return 0.0
    return len(re.findall(r'[\u0900-\u097F]', text)) / len(text)
def response_length(text): return len(text.split())
def repetition_score(text, n=4):
    words = text.split()
    if len(words) < n*2: return 1
    ngrams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
    c = Counter(ngrams)
    return c.most_common(1)[0][1] if c else 1

ft_results = []
start = time.time()
for i, ex in enumerate(eval_set):
    resp = gen(ex['instruction'], max_new_tokens=512)
    ft_results.append({
        'instruction': ex['instruction'],
        'eval_category': ex['eval_category'],
        'reference_output': ex.get('output', ''),
        'ft_response': resp,
        'ft_hindi_ratio': hindi_ratio(resp),
        'ft_length': response_length(resp),
        'ft_repetition': repetition_score(resp),
    })
    if (i+1) % 20 == 0:
        print(f"{i+1}/320 done ({time.time()-start:.0f}s)", flush=True)

print(f"\nGenerated {len(ft_results)} in {time.time()-start:.0f}s")

# Save immediately
with open('ft_results.json', 'w', encoding='utf-8') as f:
    json.dump(ft_results, f, ensure_ascii=False, indent=2)
print("Saved ft_results.json")

20/320 done (312s)
40/320 done (716s)
60/320 done (1032s)
80/320 done (1374s)
100/320 done (1707s)
120/320 done (2091s)
140/320 done (2326s)
160/320 done (2757s)
180/320 done (3045s)
200/320 done (3445s)
220/320 done (3709s)
240/320 done (4088s)
260/320 done (4349s)
280/320 done (4734s)
300/320 done (5051s)
320/320 done (5498s)

Generated 320 in 5498s
Saved ft_results.json
